# Create Sigrid Jusélius Foundation Awards (Sigrid Juséliuksen Säätiö)

Creates Sigrid Jusélius Foundation awards from the grant-decision PDF lists
the foundation publishes on its grant pages
(https://www.sigridjuselius.fi/en/apurahat/...). Finland's largest private
funder of medical research; every grant in these lists is medical research.
**1,714 awards, 2012-2026, EUR 242.9M** (local run 2026-10-01):

| scheme | source | awards |
|---|---|---|
| 1-3-year grants (senior researchers / young group leaders; 2012-2014 'domestic' and Homing grants) | 26 annual PDFs 2012-2026 | 1,643 |
| Senior researcher / senior clinical researcher posts (3-4 years) | cumulative PDF 2016-2025 | 61 |
| Large grants (5 years) | cumulative PDF 2013-2021 | 10 |

**How the 1-3-year lists become awards (done in the script):** each list row
is one ANNUAL instalment, marked '(k/N)' (year k of an N-year grant).
Instalments are chained per lead applicant across consecutive years into one
award; `amount` = sum of the instalments listed (so a grant still running is
the amount granted so far). 22 chains whose first instalment predates the
2012 list ship with `amount` NULL. Parsed counts and EUR totals were checked
against the counts and column totals printed in each PDF (2 PDFs print a
count one higher than their rows; their EUR totals match exactly). The
grant period is 1 May - 30 April (stated on the grant page), so
`start_date` = first year-05-01 and `end_date` = (first year + N)-04-30;
2012-2014 rows print no N, so their end_date is NULL. Senior posts carry
only the decision date (December; post starts the next year).

**No project titles are published.** `display_name` is composed:
'<scheme>: <lead> (<year>)'; `description` lists the instalments and the
institution.

**Not ingested (follow-up):** Fellowship, Senior Fellowship and Visiting
Professor cumulative lists (free-text multi-line entries) and the four named
professorships (news posts only).

**Prerequisites:**
- Run `scripts/local/sigrid_juselius_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/sigrid_juselius/sigrid_juselius_projects.parquet`
  (1.4 shrink guard).

**`funder_award_id` (2.1.1):** the foundation publishes no grant number.
Citing works write a 6-digit application number (e.g. `230181`; ~240 of the
~965 acknowledgement shell rows for this funder), which the lists never
show, so those shells cannot be collapsed. Synthetic key
`SJS-<first year>-<scheme code>-<lead family>-<lead given>` (ASCII slug;
3 ids carry a '-b' suffix where the source mis-numbers an instalment).
Zero existing shells use this form.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320323760`
- display_name / ror_id / doi: from `openalex.funders.funders` (ror 00ckakm23, doi 10.13039/501100006306)

**Priority:** `524` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.sigrid_juselius_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/sigrid_juselius/sigrid_juselius_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_awards FROM openalex.awards.sigrid_juselius_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.sigrid_juselius_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.sigrid_juselius_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320323760 must have exactly one openalex.funders.funders row') AS funder_ok,
       MAX(display_name) AS display_name, MAX(ror_id) AS ror_id, MAX(doi) AS doi
FROM openalex.funders.funders
WHERE funder_id = 4320323760;

## Step 2: Create Sigrid Jusélius Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.sigrid_juselius_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320323760  -- Sigrid Juséliuksen Säätiö
),

people AS (
    -- struct field order MUST match openalex_awards_raw: given, family, orcid, role_start, affiliation.
    SELECT
        funder_award_id,
        TRANSFORM(
            FILTER(from_json(people, 'array<struct<name:string,given_name:string,family_name:string>>'),
                   p -> p.family_name IS NOT NULL),
            p -> named_struct(
                'given_name', NULLIF(TRIM(p.given_name), ''),
                'family_name', NULLIF(TRIM(p.family_name), ''),
                'orcid', CAST(NULL AS STRING),
                'role_start', CAST(NULL AS DATE),
                'affiliation', named_struct(
                    -- bilingual 'Helsingin yliopisto / Helsingfors universitet': keep the first form
                    'name', NULLIF(TRIM(regexp_extract(institution, '^([^/]+)', 1)), ''),
                    'country', CASE WHEN institution IS NOT NULL THEN 'FI' END,
                    'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                )
            )) AS investigators
    FROM openalex.awards.sigrid_juselius_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    'EUR' as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN g.scheme_code IN ('SRP', 'SCRP') THEN 'fellowship' ELSE 'research' END as funding_type,
    g.funder_scheme as funder_scheme,
    'sigrid_juselius_grant_lists' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
    TRY_CAST(g.start_year AS INT) as start_year,
    YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
    -- try_element_at: safe on empty arrays (2.3)
    try_element_at(p.investigators, 1) as lead_investigator,
    try_element_at(p.investigators, 2) as co_lead_investigator,
    CASE WHEN size(p.investigators) > 0 THEN p.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.sigrid_juselius_raw g
JOIN people p ON p.funder_award_id = g.funder_award_id
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

## Shape check (2.1.1)

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 0, 'unexpected funder_award_id shape') AS shape_ok
FROM openalex.awards.sigrid_juselius_awards
WHERE NOT funder_award_id RLIKE '^SJS-(19|20)[0-9]{2}-(SR|YG|DG|HG|SRP|SCRP|LG)-[a-z0-9-]+$';

In [ ]:
%sql
SELECT assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok, COUNT(*) AS n
FROM openalex.awards.sigrid_juselius_awards;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'sigrid_juselius_grant_lists' AND priority = 524;

-- Priority 524: direct-from-funder ingest of the Sigrid Jusélius Foundation's grant lists.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    524 as priority
FROM openalex.awards.sigrid_juselius_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.sigrid_juselius_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, COUNT(amount) as with_amount, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.sigrid_juselius_awards
GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.sigrid_juselius_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.sigrid_juselius_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator) as has_pi,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.sigrid_juselius_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'sigrid_juselius_grant_lists'
GROUP BY provenance, priority;